In [1]:
import cv2
import uiautomator2 as u2
import numpy as np
from PIL import Image
import io
import time
from games.water_color_sorting import state
import games.water_color_sorting.analyst as analyst
import screens

In [2]:
d = u2.connect("R9YR810XVMX")  # Connect to device

In [3]:
# Open the android application
# Todo make sure the phone is unlocked
package_name = "com.stacity.sort.color.water.drink" #game application
d.app_start(package_name)

time.sleep(20)

In [4]:
screenshot = d.screenshot()
screenshot.save('games/water_color_sorting/temp/game_screen.png')

In [5]:
# Load and process the image
img = cv2.imread('games/water_color_sorting/temp/game_screen.png')

# Convert to HSV for better color detection
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# Define color range for typical play button colors (yellow/blue)
lower_color = np.array([20, 100, 100])  # Adjusted for yellow
upper_color = np.array([30, 255, 255])  # Adjusted for yellow

# Create mask for color detection
mask = cv2.inRange(hsv, lower_color, upper_color)

# Find contours of potential buttons
contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

if contours:
    # Find the largest contour (likely to be the button)
    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)
    
    # Calculate center of the button
    center_x = x + w//2
    center_y = y + h//2
    
    print(f"{center_x =} {center_y =}")
    d.click(center_x, center_y)
else:
    print("No button detected with current color range")

time.sleep(5)

center_x =360 center_y =921


In [6]:
def solve_game(img_path: str):
    d.screenshot().save(img_path)
    game_state = state.extract_from_image(img_path)
    reduced_game_state = analyst.generate_reduced_game_state(game_state)
    solution = analyst.Solver(reduced_game_state).solve()
    for move in solution:
        # print(move)
        select_tube = game_state.tubes[move.select]
        target_tube = game_state.tubes[move.target]

        time.sleep(1)
        d.click(select_tube.position.x, select_tube.position.y)
        time.sleep(1)
        d.click(target_tube.position.x, target_tube.position.y)
        time.sleep(1)

    time.sleep(10)

In [7]:
def click_position(x_position:int =360, y_position:int = 1160):
    d.click(x_position, y_position)
    time.sleep(3.5)


In [8]:
path = 'games/water_color_sorting/temp/current_screen.png'
while True:
    current_screen = d.screenshot().save(path)
    screen_type = screens.detect_screen_type(path)
    print(screen_type)
    if screen_type == screens.ScreenType.IN_GAME: solve_game(path)
    if screen_type == screens.ScreenType.GAME_WON: click_position()
    if screen_type == screens.ScreenType.FILLING_PROGRESSION: click_position()
    if screen_type == screens.ScreenType.PROGRESSION_COMPLETE: click_position(360, 1250)
    if screen_type == screens.ScreenType.UNKNOWN: break

ScreenType.IN_GAME
ScreenType.GAME_WON
ScreenType.FILLING_PROGRESSION
ScreenType.FILLING_PROGRESSION
ScreenType.IN_GAME
ScreenType.IN_GAME
{'tubes': {'tube 1': {'position': {'x': 110, 'y': 739}, 'color_1': {'color_index': 0, 'name': 'LightPink', 'rgb_code': {'r': 255, 'g': 182, 'b': 193}, 'position': {'x': 110, 'y': 673}}, 'color_2': {'color_index': 0, 'name': 'LawnGreen', 'rgb_code': {'r': 124, 'g': 252, 'b': 0}, 'position': {'x': 110, 'y': 721}}, 'color_3': {'color_index': 0, 'name': 'LawnGreen', 'rgb_code': {'r': 124, 'g': 252, 'b': 0}, 'position': {'x': 110, 'y': 772}}, 'color_4': {'color_index': 0, 'name': 'LawnGreen', 'rgb_code': {'r': 124, 'g': 252, 'b': 0}, 'position': {'x': 110, 'y': 820}}}, 'tube 2': {'position': {'x': 210, 'y': 738}, 'color_1': {'color_index': 0, 'name': 'LightPink', 'rgb_code': {'r': 255, 'g': 182, 'b': 193}, 'position': {'x': 210, 'y': 672}}, 'color_2': {'color_index': 0, 'name': 'GameLavender', 'rgb_code': {'r': 254, 'g': 186, 'b': 253}, 'position': {'x':

Exception: State is not valid. Read errors above: ☝️